In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/camnugent/california-housing-prices/housing.csv


In [3]:
import pandas as pd
import numpy as np

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.feature_selection import VarianceThreshold

In [4]:
housing = fetch_california_housing(as_frame=True)

df = housing.frame.copy()

print(df.head())
print("\nShape:", df.shape)

   MedInc  HouseAge  AveRooms  AveBedrms  Population  AveOccup  Latitude  \
0  8.3252      41.0  6.984127   1.023810       322.0  2.555556     37.88   
1  8.3014      21.0  6.238137   0.971880      2401.0  2.109842     37.86   
2  7.2574      52.0  8.288136   1.073446       496.0  2.802260     37.85   
3  5.6431      52.0  5.817352   1.073059       558.0  2.547945     37.85   
4  3.8462      52.0  6.281853   1.081081       565.0  2.181467     37.85   

   Longitude  MedHouseVal  
0    -122.23        4.526  
1    -122.22        3.585  
2    -122.24        3.521  
3    -122.25        3.413  
4    -122.25        3.422  

Shape: (20640, 9)


In [5]:
print("\nMissing Values")
print(df.isnull().sum())

print("\nSummary Statistics")
print(df.describe())


Missing Values
MedInc         0
HouseAge       0
AveRooms       0
AveBedrms      0
Population     0
AveOccup       0
Latitude       0
Longitude      0
MedHouseVal    0
dtype: int64

Summary Statistics
             MedInc      HouseAge      AveRooms     AveBedrms    Population  \
count  20640.000000  20640.000000  20640.000000  20640.000000  20640.000000   
mean       3.870671     28.639486      5.429000      1.096675   1425.476744   
std        1.899822     12.585558      2.474173      0.473911   1132.462122   
min        0.499900      1.000000      0.846154      0.333333      3.000000   
25%        2.563400     18.000000      4.440716      1.006079    787.000000   
50%        3.534800     29.000000      5.229129      1.048780   1166.000000   
75%        4.743250     37.000000      6.052381      1.099526   1725.000000   
max       15.000100     52.000000    141.909091     34.066667  35682.000000   

           AveOccup      Latitude     Longitude   MedHouseVal  
count  20640.000000  2

## Rooms per Household

In [6]:
df["Rooms_per_Household"] = (
    df["AveRooms"] / df["AveOccup"]
)

## Bedrooms per Room

In [7]:
df["Bedrooms_per_Room"] = (
    df["AveBedrms"] / df["AveRooms"]
)

## Population per Household

In [8]:
df["Population_per_Household"] = (
    df["Population"] / df["AveOccup"]
)

## Income per Room

In [9]:
df["Income_per_Room"] = (
    df["MedInc"] / df["AveRooms"]
)

## Income × House Age (Interaction Feature)

In [10]:

df["Income_Age_Interaction"] = (
    df["MedInc"] * df["HouseAge"]
)

## Location Interaction

In [11]:
df["Latitude_Longitude"] = (
    df["Latitude"] * df["Longitude"]
)

## Log Transformation (Population)

In [13]:
df["Population_Log"] = np.log1p(df["Population"])

## Density Feature

In [14]:
df["Population_Density"] = (
    df["Population"] /
    (df["AveRooms"] + 1)
)

## Scaling

In [15]:
numerical_columns = df.drop(columns=["MedHouseVal"]).columns

scaler = StandardScaler()

df[numerical_columns] = scaler.fit_transform(
    df[numerical_columns]
)

print("\nScaling Completed")


Scaling Completed


## Polynomial Features

In [16]:

poly = PolynomialFeatures(
    degree=2,
    include_bias=False
)

poly_features = poly.fit_transform(
    df[["MedInc", "HouseAge"]]
)

poly_columns = poly.get_feature_names_out(
    ["MedInc", "HouseAge"]
)

poly_df = pd.DataFrame(
    poly_features,
    columns=poly_columns
)

print("\nPolynomial Features")
print(poly_df.head())


Polynomial Features
     MedInc  HouseAge  MedInc^2  MedInc HouseAge  HouseAge^2
0  2.344766  0.982143  5.497926         2.302894    0.964604
1  2.332238 -0.607019  5.439334        -1.415713    0.368472
2  1.782699  1.856182  3.178017         3.309014    3.445410
3  0.932968  1.856182  0.870428         1.731757    3.445410
4 -0.012881  1.856182  0.000166        -0.023909    3.445410


## Variance Threshold Feature Selection

In [17]:
X = df.drop(columns=["MedHouseVal"])

selector = VarianceThreshold(
    threshold=0.01
)

X_selected = selector.fit_transform(X)

selected_features = X.columns[
    selector.get_support()
]

print("\nSelected Features")
print(selected_features.tolist())


Selected Features
['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude', 'Rooms_per_Household', 'Bedrooms_per_Room', 'Population_per_Household', 'Income_per_Room', 'Income_Age_Interaction', 'Latitude_Longitude', 'Population_Log', 'Population_Density']


##  Final Dataset

In [18]:
print("\nFinal Dataset Shape:", df.shape)

print("\nEngineered Features Added:")

new_features = [
    "Rooms_per_Household",
    "Bedrooms_per_Room",
    "Population_per_Household",
    "Income_per_Room",
    "Income_Age_Interaction",
    "Latitude_Longitude",
    "Population_Log",
    "Population_Density"
]

for feature in new_features:
    print("-", feature)


Final Dataset Shape: (20640, 17)

Engineered Features Added:
- Rooms_per_Household
- Bedrooms_per_Room
- Population_per_Household
- Income_per_Room
- Income_Age_Interaction
- Latitude_Longitude
- Population_Log
- Population_Density
